## Assignment: Fine-Tuning BERT for POS Tagging & Chunking

### Objective
#### Build and fine-tune a transformer model (BERT/DistilBERT) to perform Part-of-Speech (POS) Tagging and Chunking (Phrase Detection) using token classification techniques.


## Tools and Technologies
### Students should use the following:
### Python
### ugging Face Transformers
### TensorFlow or PyTorch
### Jupyter Notebook / VS Code


## Task Description
### You are required to build a token classification system using a transformer model to perform POS tagging and chunking. The task includes dataset handling, preprocessing, training, evaluation, and inference.

## Dataset Requirement
Choose any one dataset:
CoNLL-2003 (for Chunking);
Universal Dependencies (for POS Tagging);
WikiANN (Optional).


In [4]:
!pip install -q transformers datasets evaluate seqeval accelerate torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00


In [28]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
    set_seed
)

from seqeval.metrics import classification_report

# set seed for reproducibility
set_seed(42)

In [29]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


Task 1: Dataset Selection (10%)
Select a dataset
Identify label types (POS tags / Chunk tags)
Deliverables:
Dataset name
Label categories


In [30]:
import os
import shutil
from datasets import disable_caching, load_dataset

# Try to remove the cache directory to ensure a clean download
cache_dir = os.path.expanduser("~/.cache/huggingface/datasets")
if os.path.exists(cache_dir):
    print(f"Removing existing dataset cache: {cache_dir}")
    shutil.rmtree(cache_dir)
else:
    print(f"Dataset cache not found at: {cache_dir}")

disable_caching()
dataset = load_dataset("wikiann", "en", download_mode="force_redownload")

print("✅ Dataset loaded: WikiANN (English)")

# check sample
print(dataset)
print("\nSample data:")
print(dataset["train"][0])

Removing existing dataset cache: /root/.cache/huggingface/datasets


en/validation-00000-of-00001.parquet:   0%|          | 0.00/748k [00:00<?, ?B/s]

en/test-00000-of-00001.parquet:   0%|          | 0.00/748k [00:00<?, ?B/s]

en/train-00000-of-00001.parquet:   0%|          | 0.00/1.50M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

✅ Dataset loaded: WikiANN (English)
DatasetDict({
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 20000
    })
})

Sample data:
{'tokens': ['R.H.', 'Saunders', '(', 'St.', 'Lawrence', 'River', ')', '(', '968', 'MW', ')'], 'ner_tags': [3, 4, 0, 3, 4, 4, 0, 0, 0, 0, 0], 'langs': ['en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en'], 'spans': ['ORG: R.H. Saunders', 'ORG: St. Lawrence River']}


Task 2: Data Preprocessing (15%)
Tokenize using BERT tokenizer
Align labels with tokens
Handle:
Subwords
Special tokens (-100)
Expected Output:
input_ids
attention_mask
labels


In [31]:
label_list = dataset["train"].features["ner_tags"].feature.names
num_labels = len(label_list)

print("Labels:", label_list)
print("Number of labels:", num_labels)

Labels: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']
Number of labels: 7


In [32]:
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

Task 3: Model Setup (15%)
Use BERT or DistilBERT
Use AutoModelForTokenClassification
Requirements:
Correct num_labels
Proper label mapping (id2label, label2id)


In [33]:
model_checkpoint = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

print(f"Loaded tokenizer: {model_checkpoint}")

Loaded tokenizer: distilbert-base-uncased


Task 4: Training (20%)
Use Hugging Face Trainer
Define:
Learning rate
Epochs
Batch size
Train model on selected dataset


In [34]:
example = dataset["train"][0]

tokenized_input = tokenizer(example["tokens"], is_split_into_words=True)

tokens = tokenizer.convert_ids_to_tokens(tokenized_input["input_ids"])
word_ids = tokenized_input.word_ids()

print("Tokens:", tokens)
print("Word IDs:", word_ids)

Tokens: ['[CLS]', 'r', '.', 'h', '.', 'saunders', '(', 'st', '.', 'lawrence', 'river', ')', '(', '96', '##8', 'mw', ')', '[SEP]']
Word IDs: [None, 0, 0, 0, 0, 1, 2, 3, 3, 4, 5, 6, 7, 8, 8, 9, 10, None]


In [35]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i in range(len(examples["tokens"])):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        previous_word_idx = None

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(examples["ner_tags"][i][word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [36]:
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

In [37]:
print(tokenized_dataset["train"][0])

{'tokens': ['R.H.', 'Saunders', '(', 'St.', 'Lawrence', 'River', ')', '(', '968', 'MW', ')'], 'ner_tags': [3, 4, 0, 3, 4, 4, 0, 0, 0, 0, 0], 'langs': ['en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en'], 'spans': ['ORG: R.H. Saunders', 'ORG: St. Lawrence River'], 'input_ids': [101, 1054, 1012, 1044, 1012, 15247, 1006, 2358, 1012, 5623, 2314, 1007, 1006, 5986, 2620, 12464, 1007, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [-100, 3, -100, -100, -100, 4, 0, 3, -100, 4, 4, 0, 0, 0, -100, 0, 0, -100]}


In [38]:
small_train = tokenized_dataset["train"].shuffle(seed=42).select(range(2000))
small_eval = tokenized_dataset["validation"].select(range(500))

print("Reduced dataset ready")

Reduced dataset ready


In [39]:
print(tokenized_dataset)

print("\nKeys in one sample:")
print(tokenized_dataset["train"][0].keys())

DatasetDict({
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 20000
    })
})

Keys in one sample:
dict_keys(['tokens', 'ner_tags', 'langs', 'spans', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'])


In [40]:
sample = tokenized_dataset["train"][0]

print("Input IDs:", sample["input_ids"][:10])
print("Labels:", sample["labels"][:10])

Input IDs: [101, 1054, 1012, 1044, 1012, 15247, 1006, 2358, 1012, 5623]
Labels: [-100, 3, -100, -100, -100, 4, 0, 3, -100, 4]


In [41]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [42]:
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Task 5: Evaluation (15%)
Use seqeval metric
Report:
Precision
Recall
F1 Score


In [43]:
import evaluate

metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (-100) and map to label names
    true_predictions = [
        [label_list[pred] for (pred, lab) in zip(prediction, label) if lab != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[lab] for (pred, lab) in zip(prediction, label) if lab != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [44]:
args = TrainingArguments(
    output_dir="ner_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=50,
)

In [45]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=small_train,
    eval_dataset=small_eval,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [46]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [47]:
train_result = trainer.train()

print("\nTraining completed!")

# Save model
trainer.save_model("ner_model")

# Evaluate after training
eval_results = trainer.evaluate()

print("\nEvaluation Results:")
for key, value in eval_results.items():
    print(f"{key}: {value:.4f}")

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.921112,0.696783,0.447115,0.555224,0.495340,0.801882


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training completed!


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Evaluation Results:
eval_loss: 0.6968
eval_precision: 0.4471
eval_recall: 0.5552
eval_f1: 0.4953
eval_accuracy: 0.8019
eval_runtime: 2.0025
eval_samples_per_second: 249.6840
eval_steps_per_second: 15.9800
epoch: 1.0000


In [48]:
eval_results = trainer.evaluate()

print("\nEvaluation Results:")
for key, value in eval_results.items():
    print(f"{key}: {value:.4f}")


Evaluation Results:
eval_loss: 0.6968
eval_precision: 0.4471
eval_recall: 0.5552
eval_f1: 0.4953
eval_accuracy: 0.8019
eval_runtime: 2.1314
eval_samples_per_second: 234.5870
eval_steps_per_second: 15.0140
epoch: 1.0000


Task 6: Inference (10%)
Load trained model
Predict on custom sentences
Example:
 Input: John works at Google in California
Output:
POS Tags
Chunk Tags


In [49]:
text = "John works at Google in California"

tokens = text.split()

inputs = tokenizer(
    tokens,
    is_split_into_words=True,
    return_tensors="pt"
).to(device)

model.to(device)
model.eval()

with torch.no_grad():
    outputs = model(**inputs).logits

predictions = torch.argmax(outputs, dim=-1)

predicted_labels = [label_list[p.item()] for p in predictions[0]]

print("\nInference Result:\n")
for token, label in zip(tokens, predicted_labels):
    print(f"{token} → {label}")


Inference Result:

John → O
works → B-PER
at → O
Google → O
in → I-ORG
California → I-ORG


Task 7: Comparison (10%)
Compare:
POS Tagging → Grammar-level tagging (Easy)
Chunking → Phrase-level grouping (Medium)


In [54]:
from seqeval.metrics import classification_report

predictions, labels, _ = trainer.predict(small_eval)

preds = np.argmax(predictions, axis=2)

true_predictions = [
    [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(preds, labels)
]

true_labels = [
    [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(preds, labels)
]

print(classification_report(true_labels, true_predictions))

              precision    recall  f1-score   support

         LOC       0.43      0.44      0.43       203
         ORG       0.30      0.41      0.35       225
         PER       0.61      0.79      0.69       242

   micro avg       0.45      0.56      0.50       670
   macro avg       0.45      0.55      0.49       670
weighted avg       0.45      0.56      0.50       670



In [55]:
trainer.save_model("final_ner_model")
tokenizer.save_pretrained("final_ner_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('final_ner_model/tokenizer_config.json', 'final_ner_model/tokenizer.json')

Task 8: Report / Blog (5%)
Write about:
Differences between POS Tagging and Chunking
Challenges faced
Observations and insights


In [56]:
print("Model Labels Mapping:")
print(id2label)

Model Labels Mapping:
{0: 'O', 1: 'B-PER', 2: 'I-PER', 3: 'B-ORG', 4: 'I-ORG', 5: 'B-LOC', 6: 'I-LOC'}


Task 7: Comparison
## 🔍 Comparison: POS Tagging vs Chunking

### Part-of-Speech (POS) Tagging
- Assigns grammatical labels to each word
- Example: Noun, Verb, Adjective
- Focuses on syntax (grammar structure)
- Easier task because it is word-level classification

### Chunking (Phrase Detection)
- Groups words into meaningful phrases
- Example: Noun Phrase (NP), Verb Phrase (VP)
- Focuses on sentence structure and context
- More complex than POS tagging

### Key Differences

| Feature        | POS Tagging              | Chunking                     |
|---------------|--------------------------|------------------------------|
| Level         | Word-level               | Phrase-level                 |
| Complexity    | Easy                     | Medium                       |
| Purpose       | Grammar identification   | Phrase grouping              |
| Example       | Noun, Verb               | Noun Phrase, Verb Phrase     |

### Conclusion
POS tagging helps understand grammar, while chunking helps understand how words form meaningful phrases in a sentence.

Task 8: Report
## 📝 Report: Insights from the Project

### Differences between POS Tagging and Chunking
POS tagging focuses on identifying the grammatical role of each word, while chunking focuses on grouping words into meaningful phrases. Chunking requires understanding context, making it slightly more complex.

### Challenges Faced
- Handling subword tokenization using BERT tokenizer
- Aligning labels correctly with tokens
- Managing special tokens using -100
- Training time issues on CPU (resolved using GPU and smaller dataset)

### Observations and Insights
- Transformer models like DistilBERT provide better contextual understanding
- Label alignment is the most critical step in token classification
- Even small datasets can give good results for demonstration purposes
- Evaluation metrics like F1 score help measure model performance effectively

### Real-world Applications
- Chatbots and virtual assistants
- Resume parsing systems
- Information extraction from documents
- Healthcare and finance NLP systems